# Modul 6: Integrasi Teknik AI dalam Sistem Informasi
## Studi Kasus 2: Sistem Informasi Kesehatan Kampus (Klasifikasi Tingkat Risiko Stres Mahasiswa)

### Deskripsi Studi Kasus & Arsitektur Komponen
Notebook ini mengimplementasikan arsitektur berlapis (*Layered Architecture*) untuk mengklasifikasikan tingkat risiko stres mahasiswa (`0` = Risiko Rendah, `1` = Risiko Sedang, `2` = Risiko Tinggi) dari hasil kuesioner harian:
1. **Model Layer (Fase Training & Serialisasi):** Melatih model klasifikasi multi-kelas `MLPClassifier` dengan `StandardScaler` dan menyimpannya ke `model_stres.joblib`.
2. **Data Layer (SQLite3):** Menyimpan riwayat kuesioner, tingkat risiko stres, serta stempel waktu ke dalam tabel `log_stres` pada `log_kesehatan.db`.
3. **Application Layer (`templates/index.html`):** Antarmuka web skrining mandiri untuk mahasiswa.
4. **API Layer (Flask):** Menyediakan endpoint REST `/predict` (POST) untuk melayani klasifikasi secara *real-time* (Fase Serving).
5. **Pengujian End-to-End:** Menguji endpoint pada ketiga kelas tingkat stres menggunakan `requests` dan memverifikasi pencatatan log SQLite.

### 1. Instalasi & Impor Pustaka

In [1]:
# Jalankan baris di bawah jika pustaka belum terpasang:
# !pip install scikit-learn joblib flask requests numpy

import os
import sqlite3
import threading
import time
from datetime import datetime
import joblib
import numpy as np
import requests
from flask import Flask, jsonify, render_template, request
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

### 2. Model Layer: Pelatihan & Serialisasi Model (`train_model.py`)
* **Fitur Input:** `[jam_tidur, beban_sks, jam_kerja_paruh_waktu]`
* **Target Label:** `0` = Risiko Rendah, `1` = Risiko Sedang, `2` = Risiko Tinggi

In [2]:
# Dataset historis hasil kuesioner kesehatan mahasiswa
# Fitur: [jam_tidur_per_hari, beban_sks_semester, jam_kerja_paruh_waktu_per_minggu]
X = np.array([
    [8.0, 18, 0],   # Tidur cukup, SKS normal, tidak kerja -> Risiko Rendah (0)
    [7.5, 20, 4],   # Tidur cukup, SKS normal, kerja ringan -> Risiko Rendah (0)
    [8.5, 16, 0],   # Istirahat sangat baik -> Risiko Rendah (0)
    [7.0, 19, 5],   # Pola seimbang -> Risiko Rendah (0)
    [6.0, 22, 10],  # Tidur agak kurang, SKS padat, kerja paruh waktu -> Risiko Sedang (1)
    [5.5, 20, 12],  # Tidur kurang, beban kerja sedang -> Risiko Sedang (1)
    [6.5, 24, 8],   # SKS maksimal, tidur cukup -> Risiko Sedang (1)
    [6.0, 21, 14],  # Tekanan waktu mulai tinggi -> Risiko Sedang (1)
    [4.0, 24, 20],  # Kurang tidur parah, SKS penuh, kerja berat -> Risiko Tinggi (2)
    [4.5, 22, 25],  # Kelelahan fisik & akademik -> Risiko Tinggi (2)
    [3.5, 24, 18],  # Defisit tidur ekstrem -> Risiko Tinggi (2)
    [5.0, 23, 22],  # Beban ganda berat -> Risiko Tinggi (2)
])

y = np.array([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2])

# Membagi data latih dan data uji (75% latih, 25% uji) dengan stratifikasi kelas
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Pipeline: StandardScaler + MLPClassifier
model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(10, 6), activation="relu", max_iter=2500, random_state=42)
)

# Melatih model
model.fit(X_train, y_train)
print(f"Akurasi model pada data uji: {model.score(X_test, y_test):.2f}")

# Serialisasi model ke file fisik
nama_file_model = "model_stres.joblib"
joblib.dump(model, nama_file_model)
print(f"Model berhasil disimpan sebagai '{nama_file_model}'")

Akurasi model pada data uji: 0.67
Model berhasil disimpan sebagai 'model_stres.joblib'


### 3. Application Layer: Membuat Antarmuka Web (`templates/index.html`)

In [3]:
os.makedirs("templates", exist_ok=True)

html_content = '''<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Skrining Risiko Stres Mahasiswa - Kesehatan Kampus</title>
    <style>
        body { font-family: sans-serif; max-width: 480px; margin: 40px auto; line-height: 1.6; }
        .form-group { margin-bottom: 14px; }
        label { display: block; font-weight: bold; margin-bottom: 4px; }
        input, button { width: 100%; padding: 8px; box-sizing: border-box; }
        button { background: #111; color: #fff; border: none; cursor: pointer; font-weight: bold; margin-top: 8px; }
        button:hover { background: #333; }
        #hasil { margin-top: 20px; padding: 12px; border: 1px solid #ccc; display: none; }
    </style>
</head>
<body>
    <h2>Kuesioner Tingkat Risiko Stres Mahasiswa</h2>
    <p>Isi data aktivitas harian Anda untuk mengetahui tingkat risiko stres saat ini.</p>

    <div class="form-group">
        <label for="jam_tidur">Rata-Rata Jam Tidur per Hari:</label>
        <input type="number" id="jam_tidur" step="0.5" min="0" max="24" placeholder="Contoh: 6.5" required>
    </div>

    <div class="form-group">
        <label for="beban_sks">Beban SKS Semester Ini:</label>
        <input type="number" id="beban_sks" min="1" max="24" placeholder="Contoh: 20" required>
    </div>

    <div class="form-group">
        <label for="jam_kerja">Jam Kerja Paruh Waktu per Minggu (0 jika tidak bekerja):</label>
        <input type="number" id="jam_kerja" step="0.5" min="0" placeholder="Contoh: 10" required>
    </div>

    <button onclick="cekStres()">Analisis Risiko Stres</button>

    <div id="hasil"></div>

    <script>
        async function cekStres() {
            const hasilDiv = document.getElementById("hasil");
            const payload = {
                jam_tidur: parseFloat(document.getElementById("jam_tidur").value),
                beban_sks: parseInt(document.getElementById("beban_sks").value),
                jam_kerja_paruh_waktu: parseFloat(document.getElementById("jam_kerja").value)
            };

            try {
                const response = await fetch("/predict", {
                    method: "POST",
                    headers: { "Content-Type": "application/json" },
                    body: JSON.stringify(payload)
                });

                const data = await response.json();
                hasilDiv.style.display = "block";

                if (response.ok) {
                    hasilDiv.innerHTML = `
                        <strong>Hasil Klasifikasi:</strong> ${data.label}<br>
                        <strong>Tingkat Keyakinan Model:</strong> ${(data.probabilitas * 100).toFixed(1)}%
                    `;
                } else {
                    hasilDiv.innerHTML = `<strong>Error:</strong> ${data.error}`;
                }
            } catch (err) {
                hasilDiv.style.display = "block";
                hasilDiv.innerHTML = `<strong>Gagal terhubung ke API:</strong> ${err.message}`;
            }
        }
    </script>
</body>
</html>'''

with open("templates/index.html", "w", encoding="utf-8") as f:
    f.write(html_content)

print("File 'templates/index.html' berhasil dibuat.")

File 'templates/index.html' berhasil dibuat.


### 4. API Layer & Data Layer (`app.py`)

In [4]:
app = Flask(__name__)

LABEL_STRES = {
    0: "Risiko Stres Rendah",
    1: "Risiko Stres Sedang",
    2: "Risiko Stres Tinggi",
}

DB_NAME = "log_kesehatan.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS log_stres (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            waktu TEXT NOT NULL,
            bulan_tahun TEXT NOT NULL,
            jam_tidur REAL NOT NULL,
            beban_sks INTEGER NOT NULL,
            jam_kerja_paruh_waktu REAL NOT NULL,
            prediksi INTEGER NOT NULL,
            label TEXT NOT NULL,
            probabilitas REAL NOT NULL
        )
    ''')
    conn.commit()
    conn.close()

def log_prediction(data, prediksi, label, probabilitas):
    sekarang = datetime.now()
    conn = sqlite3.connect(DB_NAME)
    conn.execute(
        '''
        INSERT INTO log_stres (
            waktu, bulan_tahun, jam_tidur, beban_sks,
            jam_kerja_paruh_waktu, prediksi, label, probabilitas
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''',
        (
            sekarang.isoformat(),
            sekarang.strftime("%Y-%m"),
            data["jam_tidur"],
            data["beban_sks"],
            data["jam_kerja_paruh_waktu"],
            prediksi,
            label,
            probabilitas,
        ),
    )
    conn.commit()
    conn.close()

init_db()
loaded_model = joblib.load("model_stres.joblib")

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()
    if not data:
        return jsonify({"error": "Payload JSON tidak ditemukan"}), 400

    try:
        jam_tidur = float(data["jam_tidur"])
        beban_sks = int(data["beban_sks"])
        jam_kerja = float(data["jam_kerja_paruh_waktu"])

        if not (0 <= jam_tidur <= 24) or beban_sks < 0 or jam_kerja < 0:
            return jsonify({"error": "Nilai input berada di luar batas wajar"}), 400

        fitur = np.array([[jam_tidur, beban_sks, jam_kerja]])
    except KeyError as e:
        return jsonify({"error": f"Field wajib hilang: {e}"}), 400
    except (ValueError, TypeError):
        return jsonify({"error": "Tipe data input tidak valid"}), 400

    prediksi = int(loaded_model.predict(fitur)[0])
    probabilitas = float(loaded_model.predict_proba(fitur)[0][prediksi])
    label = LABEL_STRES.get(prediksi, "Tidak Diketahui")

    log_prediction(
        {
            "jam_tidur": jam_tidur,
            "beban_sks": beban_sks,
            "jam_kerja_paruh_waktu": jam_kerja,
        },
        prediksi,
        label,
        round(probabilitas, 3),
    )

    return jsonify({
        "prediksi": prediksi,
        "label": label,
        "probabilitas": round(probabilitas, 3),
    })

print("Aplikasi Flask & SQLite siap.")

Aplikasi Flask & SQLite siap.


### 5. Menjalankan Server & Pengujian End-to-End (`test_api.py`)

In [6]:
PORT = 5002

def run_server():
    app.run(port=PORT, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)

URL_API = f"http://127.0.0.1:{PORT}/predict"

test_cases = [
    {"jam_tidur": 8.0, "beban_sks": 18, "jam_kerja_paruh_waktu": 0},   # Risiko Rendah
    {"jam_tidur": 6.0, "beban_sks": 22, "jam_kerja_paruh_waktu": 10},  # Risiko Sedang
    {"jam_tidur": 4.0, "beban_sks": 24, "jam_kerja_paruh_waktu": 20},  # Risiko Tinggi
]

print("=== 1. MENGUJI ENDPOINT REST API ===")
for idx, kasus in enumerate(test_cases, start=1):
    res = requests.post(URL_API, json=kasus)
    print(f"Kasus {idx}: Input={kasus} -> Status={res.status_code}, Output={res.json()}")

print("=== 2. VERIFIKASI DATA LAYER (SQLITE LOG) ===")
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()
cursor.execute("SELECT id, waktu, bulan_tahun, jam_tidur, beban_sks, jam_kerja_paruh_waktu, label, probabilitas FROM log_stres")
rows = cursor.fetchall()
conn.close()

print(f"Total baris tercatat di database: {len(rows)}")
for row in rows[-len(test_cases):]:
    print(row)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5002
Press CTRL+C to quit


127.0.0.1 - - [19/Nov/2026 19:19:18] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 19:19:18] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 19:19:18] "POST /predict HTTP/1.1" 200 -


=== 1. MENGUJI ENDPOINT REST API ===
Kasus 1: Input={'jam_tidur': 8.0, 'beban_sks': 18, 'jam_kerja_paruh_waktu': 0} -> Status=200, Output={'label': 'Risiko Stres Rendah', 'prediksi': 0, 'probabilitas': 0.998}
Kasus 2: Input={'jam_tidur': 6.0, 'beban_sks': 22, 'jam_kerja_paruh_waktu': 10} -> Status=200, Output={'label': 'Risiko Stres Sedang', 'prediksi': 1, 'probabilitas': 0.956}
Kasus 3: Input={'jam_tidur': 4.0, 'beban_sks': 24, 'jam_kerja_paruh_waktu': 20} -> Status=200, Output={'label': 'Risiko Stres Tinggi', 'prediksi': 2, 'probabilitas': 0.988}
=== 2. VERIFIKASI DATA LAYER (SQLITE LOG) ===
Total baris tercatat di database: 3
(1, '2026-11-19T19:19:18.063145', '2026-11', 8.0, 18, 0.0, 'Risiko Stres Rendah', 0.998)
(2, '2026-11-19T19:19:18.076614', '2026-11', 6.0, 22, 10.0, 'Risiko Stres Sedang', 0.956)
(3, '2026-11-19T19:19:18.089587', '2026-11', 4.0, 24, 20.0, 'Risiko Stres Tinggi', 0.988)


127.0.0.1 - - [19/Nov/2026 19:19:25] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 19:19:25] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [19/Nov/2026 19:19:39] "POST /predict HTTP/1.1" 200 -


### 6. Jawaban Pertanyaan Latihan Mandiri

1. **Fitur Input Model & Relevansinya:**
   * `jam_tidur`: Durasi istirahat harian menjadi indikator pemulihan fisik dan kognitif; defisit tidur kronis berkorelasi dengan kerentanan stres.
   * `beban_sks`: Merepresentasikan intensitas tuntutan akademik dan kepadatan tugas kuliah pada semester berjalan.
   * `jam_kerja_paruh_waktu`: Mengukur beban aktivitas tambahan di luar perkuliahan yang berpotensi memicu kelelahan (*role overload*).
2. **Perancangan Skema Tabel Log untuk Analisis Lanjutan:**
   * Tabel `log_stres` mencatat `waktu` (ISO-8601) dan `bulan_tahun` (`YYYY-MM`) untuk menganalisis tren stres mahasiswa per bulan maupun saat pekan ujian (`GROUP BY bulan_tahun, label`), serta memudahkan penyaringan mahasiswa berisiko tinggi (`WHERE prediksi = 2`) untuk tindak lanjut konseling.
3. **Risiko Jika API Model Tidak Tersedia (*Down*) & Solusinya:**
   * **Risiko:** Mahasiswa tidak memperoleh hasil skrining instan dan proses pengiriman kuesioner pada portal kesehatan kampus berisiko gagal (*timeout*).
   * **Mitigasi:** Menggunakan mekanisme *fallback* berbasis skor heuristik sederhana saat API gagal dihubungi, menyimpan antrean kuesioner untuk diproses ulang saat API pulih, serta menyediakan endpoint `/health` untuk pemantauan otomatis.